In [7]:
import math
import copy

graph = {
 "A": [{"to": "B", "points": 5, "claimed": False},
       {"to": "C", "points": 3, "claimed": False},
       {"to": "D", "points": 4, "claimed": False}],
 "B": [{"to": "A", "points": 5, "claimed": False},
       {"to": "C", "points": 2, "claimed": False},
       {"to": "D", "points": 6, "claimed": False}],
 "C": [{"to": "A", "points": 3, "claimed": False},
       {"to": "B", "points": 2, "claimed": False},
       {"to": "D", "points": 2, "claimed": False},
       {"to": "E", "points": 5, "claimed": False}],
 "D": [{"to": "A", "points": 4, "claimed": False},
       {"to": "B", "points": 6, "claimed": False},
       {"to": "C", "points": 2, "claimed": False},
       {"to": "E", "points": 6, "claimed": False}],
 "E": [{"to": "C", "points": 5, "claimed": False},
       {"to": "D", "points": 6, "claimed": False}]
}

class Route:
    def __init__(self, c1, c2, points):
        self.c1 = c1
        self.c2 = c2
        self.points = points
        self.claimed_by = None # Replaced boolean with player tracking ("MAX" or "MIN")

class GameState:
    def __init__(self, routes, scores):
        self.routes = routes
        self.scores = scores

def generate_moves(state):
    # Returns the indices of the unclaimed routes so we can apply moves easily
    return [i for i, r in enumerate(state.routes) if r.claimed_by is None]

def apply_move(state, move_ind, player):
    new_state = copy.deepcopy(state)
    route = new_state.routes[move_ind]
    route.claimed_by = player
    new_state.scores[player] += route.points
    return new_state

def evaluate(state):
    # +route points
    score_diff = state.scores["MAX"] - state.scores["MIN"]

    max_connections = {}
    min_connections = {}

    # Calculating connections to determine bonuses and penalties
    for r in state.routes:
        if r.claimed_by == "MAX":
            max_connections[r.c1] = max_connections.get(r.c1, 0) + 1
            max_connections[r.c2] = max_connections.get(r.c2, 0) + 1
        elif r.claimed_by == "MIN":
            min_connections[r.c1] = min_connections.get(r.c1, 0) + 1
            min_connections[r.c2] = min_connections.get(r.c2, 0) + 1

    bonus = 0

    # +bonus for connecting cities
    for count in max_connections.values():
        if count > 1:
            bonus += 3

    # -penalty if opponent blocks key routes
    for count in min_connections.values():
        if count > 1:
            bonus -= 3

    return score_diff + bonus

# Recursive Minimax algorithm
def minimax(state, depth, is_max):
    moves = generate_moves(state)

    if depth == 0 or not moves:
        return evaluate(state)

    if is_max:
        best = -math.inf
        for move_ind in moves:
            new_state = apply_move(state, move_ind, "MAX")
            best = max(best, minimax(new_state, depth - 1, False))
        return best
    else:
        best = math.inf
        for move_ind in moves:
            new_state = apply_move(state, move_ind, "MIN")
            best = min(best, minimax(new_state, depth - 1, True))
        return best

# Find best move for AI (MAX)
def find_best_move(state, depth):
    best_val = -math.inf
    best_move = -1
    moves = generate_moves(state)

    for move_ind in moves:
        new_state = apply_move(state, move_ind, "MAX")
        move_val = minimax(new_state, depth - 1, False)
        if move_val > best_val:
            best_val = move_val
            best_move = move_ind
    return best_move

# ================ MAIN FUNCTION ================

def main():
    # Converting bidirectional dictionary graph into a list of unique Route objects
    unique_routes = []
    seen = set()

    for city, edges in graph.items():
        for edge in edges:
            route_key = tuple(sorted([city, edge["to"]]))
            if route_key not in seen:
                seen.add(route_key)
                unique_routes.append(Route(city, edge["to"], edge["points"]))

    # Initialize Game State
    initial_scores = {"MAX": 0, "MIN": 0}
    state = GameState(unique_routes, initial_scores)

    search_depth = 3

    # AI selecting optimal move
    best_move_index = find_best_move(state, search_depth)

    print("=============== FINAL RESULTS ===============")
    if best_move_index != -1:
        best_route = state.routes[best_move_index]
        print(f"Optimal Move Index: {best_move_index}")
        print(f"The AI (MAX) optimal claim route {best_route.c1}-{best_route.c2}")
        print(f"Points Gained: {best_route.points}")
    else:
        print("No legal moves remaining")

main()

=============== FINAL RESULTS ===============
Optimal Move Index: 4
The AI (MAX) optimal claim route B-D
Points Gained: 6
